# Módulo 1 — Práctica Día 1: sintaxis esencial y estructuras de datos

Ejercicios de la práctica del **Día 1** (ver [`README.md`](../README.md)).
Todos los datos son **reales**: mediciones cada hora de 5 estaciones
automáticas de SENAMHI en Lima, marzo de 2025 (carpeta `data/senamhi/`).
No se descarga nada de internet.

## Antes de empezar — leer un archivo de SENAMHI

Cada archivo es texto: una línea por hora, con los valores separados por
`;`. Por ejemplo, la línea

```
2025;03;14;13:00;27.8;0.0;56;262;3.8
```

dice: año 2025, mes 03, día 14, a las 13:00 (hora de Perú), 27.8 °C, 0.0 mm
de lluvia, etc. La primera línea del archivo trae los nombres de las
columnas. Donde la estación no midió, dice **`S/D`** («sin dato»).

Con lo visto hoy (listas, diccionarios, bucles, funciones) ya podemos leerlo:

In [1]:
CARPETA = "../../../data/senamhi/"


def leer_senamhi_basico(archivo):
    """Lee un CSV de SENAMHI y devuelve una lista de filas; cada fila es un diccionario."""
    filas = []
    with open(CARPETA + archivo, encoding="utf-8") as f:
        encabezado = f.readline().strip().split(";")    # primera línea: nombres de columna
        for linea in f:                                   # el resto: una línea por hora
            valores = linea.strip().split(";")
            filas.append(dict(zip(encabezado, valores)))  # zip junta cada nombre con su valor
    return filas


def a_numero(texto):
    """Convierte el texto '27.8' en el número 27.8, y 'S/D' (sin dato) en None."""
    if texto == "S/D":
        return None
    return float(texto)


filas = leer_senamhi_basico("472AC278_2025_03.csv")   # Von Humboldt, La Molina
print("Número de filas (horas):", len(filas))
print("Primera fila:", filas[0])

Número de filas (horas): 743
Primera fila: {'Año': '2025', 'Mes': '03', 'Día': '01', 'Hora': '00:00', 'Temperatura (°C)': '22.3', 'Precipitación (mm)': '0.0', 'Humedad (%)': '87', 'Dir. Viento (°)': '250', 'Vel. Viento (m/s)': '0.1'}


## Ejercicio 1 — Estadísticas manuales sobre una lista (30–45 min)

Antes de usar NumPy (eso es el Día 2), calculamos a mano el promedio, el
máximo, el mínimo y cuántas horas superaron un umbral, usando solo listas,
bucles y condicionales. Los datos: las 24 temperaturas del **14 de marzo de
2025** en la estación **Von Humboldt** (La Molina). La hora es la local de
Perú.

In [2]:
temperaturas_horarias = []
for fila in filas:
    if fila["Día"] == "14":
        temperaturas_horarias.append(a_numero(fila["Temperatura (°C)"]))

print("Von Humboldt, 14/03/2025 — temperaturas horarias (°C):")
print(temperaturas_horarias)

Von Humboldt, 14/03/2025 — temperaturas horarias (°C):
[21.7, 21.5, 21.3, 20.9, 20.6, 20.3, 20.3, 20.5, 21.7, 23.8, 26.1, 27.6, 27.9, 27.8, 28.3, 29.1, 26.8, 26.0, 24.5, 22.6, 22.4, 22.2, 21.9, 21.4]


In [3]:
# Todavía no vimos Matplotlib (eso es el Día 2) -- una forma simple de
# "visualizar" sin librerías extra es una barra hecha de caracteres.
print("Temperaturas horarias — barra de caracteres (1 '█' por °C)")
for hora, valor in enumerate(temperaturas_horarias):
    barra = "█" * max(int(valor), 0)
    print(f"h{hora:02d}  {valor:5.1f} °C | {barra}")

Temperaturas horarias — barra de caracteres (1 '█' por °C)
h00   21.7 °C | █████████████████████
h01   21.5 °C | █████████████████████
h02   21.3 °C | █████████████████████
h03   20.9 °C | ████████████████████
h04   20.6 °C | ████████████████████
h05   20.3 °C | ████████████████████
h06   20.3 °C | ████████████████████
h07   20.5 °C | ████████████████████
h08   21.7 °C | █████████████████████
h09   23.8 °C | ███████████████████████
h10   26.1 °C | ██████████████████████████
h11   27.6 °C | ███████████████████████████
h12   27.9 °C | ███████████████████████████
h13   27.8 °C | ███████████████████████████
h14   28.3 °C | ████████████████████████████
h15   29.1 °C | █████████████████████████████
h16   26.8 °C | ██████████████████████████
h17   26.0 °C | ██████████████████████████
h18   24.5 °C | ████████████████████████
h19   22.6 °C | ██████████████████████
h20   22.4 °C | ██████████████████████
h21   22.2 °C | ██████████████████████
h22   21.9 °C | █████████████████████
h23   21.4 °C | 

In [4]:
def resumen_temperaturas(valores, umbral=26.0):
    suma = 0.0
    maximo = valores[0]
    minimo = valores[0]
    horas_sobre_umbral = 0

    for v in valores:
        suma += v
        if v > maximo:
            maximo = v
        if v < minimo:
            minimo = v
        if v > umbral:
            horas_sobre_umbral += 1

    promedio = suma / len(valores)
    return {
        "promedio": round(promedio, 2),
        "maximo": maximo,
        "minimo": minimo,
        "horas_sobre_umbral": horas_sobre_umbral,
    }

resumen = resumen_temperaturas(temperaturas_horarias, umbral=26.0)
print(resumen)

{'promedio': 23.63, 'maximo': 29.1, 'minimo': 20.3, 'horas_sobre_umbral': 7}


## Ejercicio 2 — Diccionarios anidados: estaciones meteorológicas (45 min)

Un diccionario con las 5 estaciones de SENAMHI: nombre, coordenadas, altura
y las 24 temperaturas del mismo día (14 de marzo).

In [5]:
# Código SENAMHI -> (nombre, latitud, longitud, altura en m)
# Copiado de data/senamhi/estaciones_lima.json
datos_estaciones = {
    "112265":   ("San Martín de Porres", -12.009, -77.084, 56),
    "112181":   ("Campo de Marte",       -12.071, -77.043, 117),
    "472AC278": ("Von Humboldt",         -12.082, -76.939, 247),
    "112278":   ("Ceres",                -12.029, -76.927, 339),
    "111290":   ("Ñaña",                 -11.987, -76.842, 553),
}

estaciones = {}
for codigo, (nombre, lat, lon, altura) in datos_estaciones.items():
    temperaturas = []
    for fila in leer_senamhi_basico(codigo + "_2025_03.csv"):
        if fila["Día"] == "14":
            temperaturas.append(a_numero(fila["Temperatura (°C)"]))
    estaciones[nombre] = {
        "lat": lat,
        "lon": lon,
        "altura": altura,
        "temperaturas": temperaturas,
    }

estaciones["Ceres"]

{'lat': -12.029,
 'lon': -76.927,
 'altura': 339,
 'temperaturas': [23.0,
  22.8,
  22.5,
  22.1,
  21.8,
  21.5,
  21.6,
  21.5,
  22.2,
  24.1,
  25.8,
  27.4,
  28.7,
  28.6,
  28.7,
  28.3,
  26.9,
  26.0,
  25.7,
  24.3,
  23.7,
  24.5,
  24.5,
  24.4]}

In [6]:
def estacion_mas_calida(estaciones):
    mejor_nombre = None
    mejor_promedio = float("-inf")
    for nombre, datos in estaciones.items():
        promedio = sum(datos["temperaturas"]) / len(datos["temperaturas"])
        if promedio > mejor_promedio:
            mejor_promedio = promedio
            mejor_nombre = nombre
    return mejor_nombre, round(mejor_promedio, 2)


def filtrar_por_altura(estaciones, altura_min, altura_max):
    return {
        nombre: datos
        for nombre, datos in estaciones.items()
        if altura_min <= datos["altura"] <= altura_max
    }


def resumen_por_estacion(estaciones):
    for nombre, datos in estaciones.items():
        promedio = sum(datos["temperaturas"]) / len(datos["temperaturas"])
        print(f"{nombre:20s} | {datos['altura']:3d} m | lat={datos['lat']:7.3f} lon={datos['lon']:7.3f} "
              f"| temp. promedio = {promedio:5.2f} °C")


nombre, promedio = estacion_mas_calida(estaciones)
print(f"Estación más cálida el 14/03: {nombre} ({promedio} °C)\n")

# En Lima todas están casi a la misma latitud: lo que cambia es la altura
lejos_del_mar = filtrar_por_altura(estaciones, altura_min=200, altura_max=600)
print("Estaciones entre 200 y 600 m de altura:", list(lejos_del_mar.keys()))
print()

resumen_por_estacion(estaciones)

Estación más cálida el 14/03: Ceres (24.61 °C)

Estaciones entre 200 y 600 m de altura: ['Von Humboldt', 'Ceres', 'Ñaña']

San Martín de Porres |  56 m | lat=-12.009 lon=-77.084 | temp. promedio = 22.65 °C
Campo de Marte       | 117 m | lat=-12.071 lon=-77.043 | temp. promedio = 22.31 °C
Von Humboldt         | 247 m | lat=-12.082 lon=-76.939 | temp. promedio = 23.63 °C
Ceres                | 339 m | lat=-12.029 lon=-76.927 | temp. promedio = 24.61 °C
Ñaña                 | 553 m | lat=-11.987 lon=-76.842 | temp. promedio = 23.68 °C


In [7]:
print("Temperatura promedio por estación — barra de caracteres, ordenado de mayor a menor")
promedios = {n: sum(d["temperaturas"]) / len(d["temperaturas"]) for n, d in estaciones.items()}
for n, p in sorted(promedios.items(), key=lambda kv: -kv[1]):
    barra = "█" * max(int(p), 0)
    print(f"{n:20s} {p:5.2f} °C | {barra}")

Temperatura promedio por estación — barra de caracteres, ordenado de mayor a menor
Ceres                24.61 °C | ████████████████████████
Ñaña                 23.68 °C | ███████████████████████
Von Humboldt         23.63 °C | ███████████████████████
San Martín de Porres 22.65 °C | ██████████████████████
Campo de Marte       22.31 °C | ██████████████████████


## Ejercicio 2b — ¿Dónde faltan datos? (15 min)

Los datos reales tienen huecos. Recorremos el mes completo de cada estación
y anotamos las horas que dicen `S/D`.

In [8]:
for codigo, (nombre, lat, lon, altura) in datos_estaciones.items():
    filas_estacion = leer_senamhi_basico(codigo + "_2025_03.csv")
    faltantes = []
    for fila in filas_estacion:
        if a_numero(fila["Temperatura (°C)"]) is None:
            faltantes.append(fila["Día"] + "/03 " + fila["Hora"])
    print(f"{nombre:20s} {len(filas_estacion)} horas leídas, {len(faltantes)} con S/D: {faltantes}")

print()
print("Marzo tiene 31 días x 24 horas =", 31 * 24, "horas")

San Martín de Porres 737 horas leídas, 0 con S/D: []
Campo de Marte       744 horas leídas, 2 con S/D: ['11/03 22:00', '13/03 23:00']
Von Humboldt         743 horas leídas, 0 con S/D: []
Ceres                737 horas leídas, 0 con S/D: []
Ñaña                 744 horas leídas, 2 con S/D: ['09/03 06:00', '13/03 22:00']

Marzo tiene 31 días x 24 horas = 744 horas


Hay **dos tipos de hueco**:

- **`S/D`**: la fila existe, pero sin valor.
- **Filas que faltan**: algunas estaciones tienen menos de 744 horas. Esas
  horas simplemente no están en el archivo, y un bucle no las «ve».

Encontrar los dos tipos antes de calcular algo es parte del trabajo con
datos reales. Lo retomamos más adelante en el curso.

## Ejercicio 3 — Clasificación de precipitación (45 min)

Función reutilizable `clasificar_precipitacion(valor)` aplicada a la lluvia
de cada día de marzo en la estación **Ñaña**. El archivo trae la lluvia de
cada hora: primero hay que sumar las horas de cada día.

In [9]:
def clasificar_precipitacion(valor_mm):
    """Clasifica un valor diario de precipitación (mm) en una categoría."""
    if valor_mm is None:
        return "sin dato"
    elif valor_mm <= 0.1:
        return "sin lluvia"
    elif valor_mm <= 2.0:
        return "llovizna"
    elif valor_mm <= 15.0:
        return "moderada"
    else:
        return "fuerte"


# Sumar la lluvia de cada hora para obtener el total de cada día
precipitacion_diaria = {}          # {"01": 0.0, "02": 0.0, ...}
for fila in leer_senamhi_basico("111290_2025_03.csv"):
    dia = fila["Día"]
    valor = a_numero(fila["Precipitación (mm)"])
    if dia not in precipitacion_diaria:
        precipitacion_diaria[dia] = 0.0
    if valor is not None:
        precipitacion_diaria[dia] += valor

for dia, valor in precipitacion_diaria.items():
    valor = round(valor, 1)        # recuerda 0.1 + 0.2 de la teoría: redondeamos
    print(f"{dia}/03: {valor:4.1f} mm -> {clasificar_precipitacion(valor)}")

01/03:  0.0 mm -> sin lluvia
02/03:  0.0 mm -> sin lluvia
03/03:  0.0 mm -> sin lluvia
04/03:  0.0 mm -> sin lluvia
05/03:  0.0 mm -> sin lluvia
06/03:  0.0 mm -> sin lluvia
07/03:  0.0 mm -> sin lluvia
08/03:  0.0 mm -> sin lluvia
09/03:  0.0 mm -> sin lluvia
10/03:  0.0 mm -> sin lluvia
11/03:  0.0 mm -> sin lluvia
12/03:  0.0 mm -> sin lluvia
13/03:  0.0 mm -> sin lluvia
14/03:  0.0 mm -> sin lluvia
15/03:  0.4 mm -> llovizna
16/03:  0.0 mm -> sin lluvia
17/03:  0.0 mm -> sin lluvia
18/03:  0.1 mm -> sin lluvia
19/03:  0.0 mm -> sin lluvia
20/03:  0.0 mm -> sin lluvia
21/03:  0.0 mm -> sin lluvia
22/03:  0.0 mm -> sin lluvia
23/03:  0.0 mm -> sin lluvia
24/03:  0.0 mm -> sin lluvia
25/03:  0.0 mm -> sin lluvia
26/03:  0.0 mm -> sin lluvia
27/03:  1.1 mm -> llovizna
28/03:  0.0 mm -> sin lluvia
29/03:  0.0 mm -> sin lluvia
30/03:  0.0 mm -> sin lluvia
31/03:  0.0 mm -> sin lluvia


In [10]:
print("Precipitación diaria en Ñaña — barra de caracteres (1 '█' por cada 0.1 mm)")
for dia, valor in precipitacion_diaria.items():
    categoria = clasificar_precipitacion(valor)
    barra = "█" * int(round(valor * 10))
    print(f"{dia}/03: {valor:4.1f} mm | {barra:12s} {categoria}")

Precipitación diaria en Ñaña — barra de caracteres (1 '█' por cada 0.1 mm)
01/03:  0.0 mm |              sin lluvia
02/03:  0.0 mm |              sin lluvia
03/03:  0.0 mm |              sin lluvia
04/03:  0.0 mm |              sin lluvia
05/03:  0.0 mm |              sin lluvia
06/03:  0.0 mm |              sin lluvia
07/03:  0.0 mm |              sin lluvia
08/03:  0.0 mm |              sin lluvia
09/03:  0.0 mm |              sin lluvia
10/03:  0.0 mm |              sin lluvia
11/03:  0.0 mm |              sin lluvia
12/03:  0.0 mm |              sin lluvia
13/03:  0.0 mm |              sin lluvia
14/03:  0.0 mm |              sin lluvia
15/03:  0.4 mm | ████         llovizna
16/03:  0.0 mm |              sin lluvia
17/03:  0.0 mm |              sin lluvia
18/03:  0.1 mm | █            sin lluvia
19/03:  0.0 mm |              sin lluvia
20/03:  0.0 mm |              sin lluvia
21/03:  0.0 mm |              sin lluvia
22/03:  0.0 mm |              sin lluvia
23/03:  0.0 mm |         

En marzo de 2025 casi no llovió en Lima: es lo normal en la costa
peruana, que es desértica. Ñaña, la estación más alta y alejada del mar,
tuvo solo 2 días con llovizna.

La categoría `"sin dato"` también es real: la estación Ceres no registró
lluvia ningún día de marzo (todas sus horas dicen `S/D`).

In [11]:
fila_ceres = leer_senamhi_basico("112278_2025_03.csv")[0]
texto = fila_ceres["Precipitación (mm)"]
print("Ceres, primera hora:", texto, "->", clasificar_precipitacion(a_numero(texto)))

Ceres, primera hora: S/D -> sin dato


## Cierre

Entregable del día: este notebook resuelto + el módulo `clasificar_precipitacion.py`
(ver celda siguiente, que lo genera como script independiente para reforzar
la diferencia entre notebook y script vista en teoría).

In [12]:
script = '''"""Clasificacion de la lluvia diaria (mm) de una estacion de SENAMHI."""
import sys


def clasificar_precipitacion(valor_mm):
    if valor_mm is None:
        return "sin dato"
    elif valor_mm <= 0.1:
        return "sin lluvia"
    elif valor_mm <= 2.0:
        return "llovizna"
    elif valor_mm <= 15.0:
        return "moderada"
    else:
        return "fuerte"


def lluvia_diaria(ruta):
    totales = {}
    with open(ruta, encoding="utf-8") as f:
        encabezado = f.readline().strip().split(";")
        for linea in f:
            fila = dict(zip(encabezado, linea.strip().split(";")))
            dia = fila["Día"]
            if dia not in totales:
                totales[dia] = 0.0
            if fila["Precipitación (mm)"] != "S/D":
                totales[dia] += float(fila["Precipitación (mm)"])
    return totales


def main():
    # Sin argumento usa Ñaña; se puede pasar otra estacion:
    #   python3 clasificar_precipitacion.py ../../../data/senamhi/112181_2025_03.csv
    if len(sys.argv) > 1:
        ruta = sys.argv[1]
    else:
        ruta = "../../../data/senamhi/111290_2025_03.csv"
    for dia, valor in lluvia_diaria(ruta).items():
        valor = round(valor, 1)
        print(f"Dia {dia}/03: {valor:4.1f} mm -> {clasificar_precipitacion(valor)}")


if __name__ == "__main__":
    main()
'''

with open("clasificar_precipitacion.py", "w", encoding="utf-8") as f:
    f.write(script)

print("Script guardado. Ejecutar desde terminal con:")
print("  python3 clasificar_precipitacion.py")

Script guardado. Ejecutar desde terminal con:
  python3 clasificar_precipitacion.py
